In [ ]:
from google.colab import drive
import os
import sys

In [ ]:
# 1. 挂载 Drive
drive.mount('/content/drive')

# 2. 切换工作目录到你的项目文件夹SiC
project_path = '/content/drive/MyDrive/SiC'
os.chdir(project_path)
sys.path.append(project_path)

Mounted at /content/drive


In [ ]:
"""
========================================================================================
Physics-Informed TVNG Bearing Fault Diagnosis Network (Speed-Informed SOTA Version)
========================================================================================

【物理先验与系统背景 (Physical Priors & System Background)】
在修改或优化此代码时，必须严格遵循以下物理和工程约束：

1. 传感器与信号源 (Sensor):
   - 采用摩擦纳米发电机 (TVNG) 信号。
   - 信号特性：包含强烈的低频静电基线漂移（需专门节点提取），以及对摩擦、高频冲击高度敏感的特性。

2. 机械系统参数 (Mechanical System):
   - 目标轴承型号：6206 深沟球轴承。
   - 采样频率 (Fs): 3200 Hz (Nyquist 频率为 1600 Hz)。
   - 运行工况 (Conditions):
     * Cond 0: 900 rpm  (转频 fr = 15.0 Hz)
     * Cond 1: 1350 rpm (转频 fr = 22.5 Hz)
     * Cond 2: 1800 rpm (转频 fr = 30.0 Hz)

3. 物理频段锚定法则 (Physical Frequency Anchoring):
   频段中心位置必须基于当前转速 (fr) 动态计算（软阶次跟踪）：
   - Node 0 [基线趋势]: 归一化频率固定在 0.001 (接近 0Hz)。
   - Node 1 [保持架 FTF]: 中心频率在 0.4 * fr。
   - Node 2 [转轴 1X2X]: 中心频率在 1.5 * fr。
   - Node 3 [故障基频群]: 内外圈故障频率代表值，设为 4.5 * fr。
   - Node 4 [高频共振带]: 结构固有属性，与转速无关，固定在归一化 0.25 (即 400 Hz)。

4. 异质带宽硬约束 (Heterogeneous Bandwidth Constraints):
   必须防止低频模态混叠和高频信息遗漏，严禁掩码退化为全通滤波器：
   - Node 0 (直流基线): 极窄带约束，Sigma 范围 [0.001, 0.002]。
   - Node 1 (保持架): 极窄带约束，Sigma 范围 [0.001, 0.005]。
   - Node 2 (转频 1X): 窄带约束，Sigma 范围 [0.002, 0.015]。
   - Node 3 (故障基频): 中带约束，Sigma 范围 [0.010, 0.040]。
   - Node 4 (高频共振): 宽带约束，Sigma 范围 [0.020, 0.080] (合理的高频包裹)。

5. 异质中心偏移约束 (Heterogeneous Shift Bounds):
   - Node 0 必须钉死在直流区，偏移容差设为 0.0。
   - Node 1, 2 给予极小容差 (0.005~0.01) 防止频段交叉。
   - Node 3, 4 给予适度容差 (0.02~0.05) 用于寻找故障与共振中心。
========================================================================================
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torch.fft
from torch.utils.data import Dataset, DataLoader
import os
import sys
import copy
import random

# ==========================================
# 随机种子设置函数 (保证多次实验的有效性与可复现性)
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# ==========================================
# 0. 数据加载器
# ==========================================
class MultiConditionDataset(Dataset):
    def __init__(self, data_paths, label_paths):
        all_data, all_labels, all_conditions = [], [], []
        for cond_idx, (d_path, l_path) in enumerate(zip(data_paths, label_paths)):
            if not os.path.exists(d_path) or not os.path.exists(l_path):
                print(f"[警告] 跳过未找到的文件: {d_path}")
                continue
            data, labels = np.load(d_path), np.load(l_path)
            if len(data.shape) == 2: data = np.expand_dims(data, axis=1)
            elif len(data.shape) == 3 and data.shape[-1] == 1: data = np.transpose(data, (0, 2, 1))
            all_data.append(data); all_labels.append(labels); all_conditions.append(np.full(labels.shape[0], cond_idx))
        if len(all_data) > 0:
            self.data = np.concatenate(all_data, axis=0)
            self.labels = np.concatenate(all_labels, axis=0)
            self.conditions = np.concatenate(all_conditions, axis=0)
        else:
            self.data, self.labels, self.conditions = np.array([]), np.array([])

    def __len__(self): return self.data.shape[0]
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long), torch.tensor(self.conditions[idx], dtype=torch.long)

# ==========================================
# 1. 梯度反转层 (GRL)
# ==========================================
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GRL(nn.Module):
    def __init__(self, alpha=1.0):
        super(GRL, self).__init__()
        self.alpha = alpha
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

# ==========================================
# 2. 动态频谱物理引导层
# ==========================================
class DynamicSpectrumDecomposition(nn.Module):
    def __init__(self, in_channels=1, out_channels=64, dropout_rate=0.15):
        super(DynamicSpectrumDecomposition, self).__init__()
        self.num_nodes = 5

        # 1. 异质化带宽边界 (防止 Node 4 全通，防止 Node 0 宽带)
        sigma_mins = [0.001, 0.001, 0.002, 0.010, 0.020]
        sigma_maxs = [0.002, 0.005, 0.015, 0.040, 0.080]
        self.register_buffer('sigma_mins', torch.tensor(sigma_mins).view(1, self.num_nodes, 1))
        ranges = [max_val - min_val for max_val, min_val in zip(sigma_maxs, sigma_mins)]
        self.register_buffer('sigma_ranges', torch.tensor(ranges).view(1, self.num_nodes, 1))

        # 2. 异质化中心偏移容差
        # Node 0 容差严格为 0；Node 1 仅给 0.005 防止挤压
        shift_bounds = [0.0, 0.005, 0.010, 0.020, 0.050]
        self.register_buffer('shift_bounds', torch.tensor(shift_bounds).view(1, self.num_nodes, 1))

        self.mu_shift = nn.Parameter(torch.zeros(1, self.num_nodes, 1))
        self.raw_widths = nn.Parameter(torch.zeros(1, self.num_nodes, 1))

        self.spectrum_cnn = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, padding=3, stride=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout1d(p=dropout_rate),

            nn.Conv1d(32, out_channels, kernel_size=3, padding=1, stride=2),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Dropout1d(p=dropout_rate),

            nn.AdaptiveMaxPool1d(1)
        )

        self.fc_node = nn.Sequential(
            nn.Linear(out_channels, 128),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate)
        )

    def forward(self, x, cond):
        X_f = torch.fft.rfft(x, dim=-1)
        amp_spectrum = torch.abs(X_f + 1e-8) / x.size(-1)

        freq_bins = amp_spectrum.size(-1)
        freq_axis = torch.linspace(0, 1, freq_bins, device=x.device)

        rpms = torch.tensor([900.0, 1350.0, 1800.0], device=x.device)
        batch_rpm = rpms[cond]
        fr = batch_rpm / 60.0
        f_nyq = 1600.0

        base_mu = torch.zeros(x.size(0), self.num_nodes, device=x.device)
        base_mu[:, 0] = 0.001                       # 1. 基线趋势 (死守直流)
        base_mu[:, 1] = 0.4 * fr / f_nyq            # 2. 保持架特征频
        base_mu[:, 2] = 1.5 * fr / f_nyq            # 3. 轴转频
        base_mu[:, 3] = 4.5 * fr / f_nyq            # 4. 故障基频群
        base_mu[:, 4] = 0.25                        # 5. 高频结构共振

        node_features = []
        for i in range(self.num_nodes):
            # 异质化偏移：Node 0 乘的是 0.0，永远在 0.001
            shift_bound_i = self.shift_bounds[:, i, :]
            mu_raw = base_mu[:, i].unsqueeze(1) + torch.tanh(self.mu_shift[:, i, :]) * shift_bound_i
            mu = torch.clamp(mu_raw, min=0.0, max=1.0)

            # 异质化带宽
            sigma_min_i = self.sigma_mins[:, i, :]
            sigma_range_i = self.sigma_ranges[:, i, :]
            sigma = torch.sigmoid(self.raw_widths[:, i, :]) * sigma_range_i + sigma_min_i

            mask = torch.exp(-0.5 * ((freq_axis - mu) / sigma)**2)
            mask = mask.unsqueeze(1)

            masked_spectrum = amp_spectrum * mask
            cnn_feat = self.spectrum_cnn(masked_spectrum).squeeze(-1)
            node_feat = self.fc_node(cnn_feat).unsqueeze(1)
            node_features.append(node_feat)

        nodes = torch.cat(node_features, dim=1)
        return nodes

# ==========================================
# 3. 图注意力机制 (GAT)
# ==========================================
class GraphAttentionLayer(nn.Module):
    def __init__(self, in_features, out_features, dropout=0.15, alpha=0.2):
        super(GraphAttentionLayer, self).__init__()
        self.W = nn.Linear(in_features, out_features, bias=False)
        self.a = nn.Linear(2 * out_features, 1, bias=False)
        self.leakyrelu = nn.LeakyReLU(alpha)
        self.dropout = nn.Dropout(dropout)

    def forward(self, h):
        B, N, F_dim = h.size()
        Wh = self.W(h)
        Wh_i = Wh.unsqueeze(2).expand(B, N, N, -1)
        Wh_j = Wh.unsqueeze(1).expand(B, N, N, -1)
        e = self.leakyrelu(self.a(torch.cat([Wh_i, Wh_j], dim=3)).squeeze(-1))

        attention = F.softmax(e, dim=-1)
        attention_drop = self.dropout(attention)

        h_prime = torch.bmm(attention_drop, Wh)
        aggregated_feature = torch.mean(h_prime, dim=1)
        return aggregated_feature, attention

# ==========================================
# 4. 原型分类器
# ==========================================
class PrototypeClassifier(nn.Module):
    def __init__(self, feature_dim, num_classes, temperature=1.0):
        super(PrototypeClassifier, self).__init__()
        self.num_classes = num_classes
        self.temperature = temperature
        self.prototypes = nn.Parameter(torch.randn(num_classes, feature_dim))
        nn.init.xavier_uniform_(self.prototypes)

    def forward(self, x):
        x_norm = F.normalize(x, p=2, dim=1)
        p_norm = F.normalize(self.prototypes, p=2, dim=1)
        cosine_sim = torch.matmul(x_norm, p_norm.t())
        logits = cosine_sim / self.temperature
        return logits, p_norm

    def get_ortho_loss(self, p_norm):
        identity = torch.eye(self.num_classes).to(p_norm.device)
        corr_matrix = torch.matmul(p_norm, p_norm.t())
        return torch.norm(corr_matrix - identity, p='fro') ** 2

# ==========================================
# 5. 域(工况)鉴别器
# ==========================================
class ConditionDiscriminator(nn.Module):
    def __init__(self, feature_dim, num_conditions=3):
        super(ConditionDiscriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_conditions)
        )

    def forward(self, x, alpha=1.0):
        x_rev = GradientReversalFunction.apply(x, alpha)
        return self.net(x_rev)

# ==========================================
# 6. 总体模型整合
# ==========================================
class TVNG_Spectrum_FaultModel(nn.Module):
    def __init__(self, num_classes=6, feature_dim=128, num_conditions=3):
        super(TVNG_Spectrum_FaultModel, self).__init__()
        self.physics_layer = DynamicSpectrumDecomposition(in_channels=1, out_channels=64, dropout_rate=0.15)
        self.gat = GraphAttentionLayer(in_features=128, out_features=feature_dim, dropout=0.15)
        self.classifier = PrototypeClassifier(feature_dim, num_classes)
        self.condition_discriminator = ConditionDiscriminator(feature_dim, num_conditions)

    def forward(self, x, cond, alpha=1.0):
        nodes = self.physics_layer(x, cond)
        features, attention_weights = self.gat(nodes)
        cls_logits, p_norm = self.classifier(features)
        cond_logits = self.condition_discriminator(features, alpha)
        return cls_logits, cond_logits, p_norm, features, attention_weights

# ==========================================
# 7. 多工况联合训练与验证主函数
# ==========================================
def train_multi_condition_model(train_x_paths, train_y_paths, val_x_paths, val_y_paths, run_idx=1):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[{device}] 正在加载多工况数据集...")

    train_dataset = MultiConditionDataset(train_x_paths, train_y_paths)
    val_dataset = MultiConditionDataset(val_x_paths, val_y_paths)

    if len(train_dataset) == 0:
        print("[错误] 未加载到任何数据！")
        return None, None

    batch_size = 64
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    num_classes = len(np.unique(train_dataset.labels))
    num_conditions = len(train_x_paths)

    model = TVNG_Spectrum_FaultModel(num_classes=num_classes, num_conditions=num_conditions).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    num_epochs = 160
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-5)

    criterion_cls = nn.CrossEntropyLoss()
    criterion_cond = nn.CrossEntropyLoss()
    lambda_ortho = 0.1
    lambda_cond = 0.4

    best_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        model.train()
        epoch_cls_loss, epoch_cond_loss = 0.0, 0.0

        for i, (data, fault_label, cond_label) in enumerate(train_loader):
            data, fault_label, cond_label = data.to(device), fault_label.to(device), cond_label.to(device)

            p = float(i + epoch * len(train_loader)) / num_epochs / len(train_loader)
            alpha = 2. / (1. + np.exp(-10 * p)) - 1

            cls_logits, cond_logits, p_norm, _, _ = model(data, cond_label, alpha)

            loss_cls = criterion_cls(cls_logits, fault_label)
            loss_cond = criterion_cond(cond_logits, cond_label)
            loss_ortho = model.classifier.get_ortho_loss(p_norm)

            total_loss = loss_cls + lambda_ortho * loss_ortho + lambda_cond * loss_cond

            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_cls_loss += loss_cls.item()
            epoch_cond_loss += loss_cond.item()

        # ---------------- 验证阶段 ----------------
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for data, fault_label, cond_label in val_loader:
                data, fault_label, cond_label = data.to(device), fault_label.to(device), cond_label.to(device)

                cls_logits, _, _, _, _ = model(data, cond_label, alpha=0.0)
                _, predicted = torch.max(cls_logits.data, 1)
                total += fault_label.size(0)
                correct += (predicted == fault_label).sum().item()

        val_acc = 100 * correct / total

        # 为了减少刷屏，每 10 个 Epoch 打印一次，或者只在最佳 Acc 出现时打印
        if (epoch + 1) % 10 == 0 or val_acc > best_acc:
            print(f"Epoch [{epoch+1:02d}/{num_epochs}] | Train Loss: {epoch_cls_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%")

        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())

        scheduler.step()

    print(f"✅ 第 {run_idx} 次训练结束！最高验证集准确率: {best_acc:.2f}%")
    model.load_state_dict(best_model_wts)

    # 动态保存每次的最佳模型
    save_path = f'best_model_run_{run_idx}.pth'
    torch.save(model.state_dict(), save_path)
    print(f"✅ 第 {run_idx} 次的最佳模型权重已保存为 '{save_path}'")
    return model, best_acc

In [ ]:
"""
========================================================================================
Physics-Informed TVNG Bearing Fault Diagnosis Network (Speed-Informed SOTA Version)
========================================================================================

【物理先验与系统背景 (Physical Priors & System Background)】
在修改或优化此代码时，必须严格遵循以下物理和工程约束：

1. 传感器与信号源 (Sensor):
   - 采用摩擦纳米发电机 (TVNG) 信号。
   - 信号特性：包含强烈的低频静电基线漂移（需专门节点提取），以及对摩擦、高频冲击高度敏感的特性。

2. 机械系统参数 (Mechanical System):
   - 目标轴承型号：6206 深沟球轴承。
   - 采样频率 (Fs): 3200 Hz (Nyquist 频率为 1600 Hz)。
   - 运行工况 (Conditions):
     * Cond 0: 900 rpm  (转频 fr = 15.0 Hz)
     * Cond 1: 1350 rpm (转频 fr = 22.5 Hz)
     * Cond 2: 1800 rpm (转频 fr = 30.0 Hz)

3. 物理频段锚定法则 (Physical Frequency Anchoring):
   频段中心位置必须基于当前转速 (fr) 动态计算（软阶次跟踪）：
   - Node 0 [基线趋势]: 归一化频率固定在 0.001 (接近 0Hz)。
   - Node 1 [保持架 FTF]: 中心频率在 0.4 * fr。
   - Node 2 [转轴 1X]: 中心频率在 1.0 * fr。
   - Node 3 [故障基频群]: 内外圈故障频率代表值，设为 4.5 * fr。
   - Node 4 [高频共振带]: 结构固有属性，与转速无关，固定在归一化 0.75 (即 1200 Hz)。

4. 异质带宽硬约束 (Heterogeneous Bandwidth Constraints):
   必须防止低频模态混叠和高频信息遗漏，严禁掩码退化为全通滤波器：
   - Node 0 (直流基线): 极窄带约束，Sigma 范围 [0.001, 0.002]。
   - Node 1 (保持架): 极窄带约束，Sigma 范围 [0.001, 0.005]。
   - Node 2 (转频 1X): 窄带约束，Sigma 范围 [0.002, 0.015]。
   - Node 3 (故障基频): 中带约束，Sigma 范围 [0.010, 0.040]。
   - Node 4 (高频共振): 宽带约束，Sigma 范围 [0.020, 0.080] (合理的高频包裹)。

5. 异质中心偏移约束 (Heterogeneous Shift Bounds):
   - Node 0 必须钉死在直流区，偏移容差设为 0.0。
   - Node 1, 2 给予极小容差 (0.005~0.01) 防止频段交叉。
   - Node 3, 4 给予适度容差 (0.02~0.05) 用于寻找故障与共振中心。
========================================================================================
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torch.fft
from torch.utils.data import Dataset, DataLoader
import os
import sys
import copy
import random
import csv  # 新增：用于保存训练日志

# ==========================================
# 随机种子设置函数 (保证多次实验的有效性与可复现性)
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# ==========================================
# 0. 数据加载器
# ==========================================
class MultiConditionDataset(Dataset):
    def __init__(self, data_paths, label_paths):
        all_data, all_labels, all_conditions = [], [], []
        for cond_idx, (d_path, l_path) in enumerate(zip(data_paths, label_paths)):
            if not os.path.exists(d_path) or not os.path.exists(l_path):
                print(f"[警告] 跳过未找到的文件: {d_path}")
                continue
            data, labels = np.load(d_path), np.load(l_path)
            if len(data.shape) == 2: data = np.expand_dims(data, axis=1)
            elif len(data.shape) == 3 and data.shape[-1] == 1: data = np.transpose(data, (0, 2, 1))
            all_data.append(data); all_labels.append(labels); all_conditions.append(np.full(labels.shape[0], cond_idx))
        if len(all_data) > 0:
            self.data = np.concatenate(all_data, axis=0)
            self.labels = np.concatenate(all_labels, axis=0)
            self.conditions = np.concatenate(all_conditions, axis=0)
        else:
            self.data, self.labels, self.conditions = np.array([]), np.array([])

    def __len__(self): return self.data.shape[0]
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long), torch.tensor(self.conditions[idx], dtype=torch.long)

# ==========================================
# 1. 梯度反转层 (GRL)
# ==========================================
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GRL(nn.Module):
    def __init__(self, alpha=1.0):
        super(GRL, self).__init__()
        self.alpha = alpha
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.alpha)

# ==========================================
# 2. 动态频谱物理引导层 (【完美物理防线版】)
# ==========================================
class DynamicSpectrumDecomposition(nn.Module):
    def __init__(self, in_channels=1, out_channels=64, dropout_rate=0.15):
        super(DynamicSpectrumDecomposition, self).__init__()
        self.num_nodes = 5

        # 1. 异质化带宽边界 (防止 Node 4 全通，防止 Node 0 宽带)
        sigma_mins = [0.001, 0.001, 0.002, 0.010, 0.020]
        sigma_maxs = [0.002, 0.005, 0.015, 0.040, 0.080]
        self.register_buffer('sigma_mins', torch.tensor(sigma_mins).view(1, self.num_nodes, 1))
        ranges = [max_val - min_val for max_val, min_val in zip(sigma_maxs, sigma_mins)]
        self.register_buffer('sigma_ranges', torch.tensor(ranges).view(1, self.num_nodes, 1))

        # 2. 异质化中心偏移容差
        # Node 0 容差严格为 0；Node 1 仅给 0.005 防止挤压
        shift_bounds = [0.0, 0.005, 0.010, 0.020, 0.050]
        self.register_buffer('shift_bounds', torch.tensor(shift_bounds).view(1, self.num_nodes, 1))

        self.mu_shift = nn.Parameter(torch.zeros(1, self.num_nodes, 1))
        self.raw_widths = nn.Parameter(torch.zeros(1, self.num_nodes, 1))

        self.spectrum_cnn = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, padding=3, stride=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout1d(p=dropout_rate),

            nn.Conv1d(32, out_channels, kernel_size=3, padding=1, stride=2),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Dropout1d(p=dropout_rate),

            nn.AdaptiveMaxPool1d(1)
        )

        self.fc_node = nn.Sequential(
            nn.Linear(out_channels, 128),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate)
        )

    def forward(self, x, cond):
        X_f = torch.fft.rfft(x, dim=-1)
        amp_spectrum = torch.abs(X_f + 1e-8) / x.size(-1)

        freq_bins = amp_spectrum.size(-1)
        freq_axis = torch.linspace(0, 1, freq_bins, device=x.device)

        rpms = torch.tensor([900.0, 1350.0, 1800.0], device=x.device)
        batch_rpm = rpms[cond]
        fr = batch_rpm / 60.0
        f_nyq = 1600.0

        base_mu = torch.zeros(x.size(0), self.num_nodes, device=x.device)
        base_mu[:, 0] = 0.001                       # 1. 基线趋势 (死守直流)
        base_mu[:, 1] = 0.4 * fr / f_nyq            # 2. 保持架特征频
        base_mu[:, 2] = 1.0 * fr / f_nyq            # 3. 轴转频
        base_mu[:, 3] = 4.5 * fr / f_nyq            # 4. 故障基频群
        base_mu[:, 4] = 0.75                        # 5. 高频结构共振

        node_features = []
        for i in range(self.num_nodes):
            # 异质化偏移：Node 0 乘的是 0.0，永远在 0.001
            shift_bound_i = self.shift_bounds[:, i, :]
            mu_raw = base_mu[:, i].unsqueeze(1) + torch.tanh(self.mu_shift[:, i, :]) * shift_bound_i
            mu = torch.clamp(mu_raw, min=0.0, max=1.0)

            # 异质化带宽
            sigma_min_i = self.sigma_mins[:, i, :]
            sigma_range_i = self.sigma_ranges[:, i, :]
            sigma = torch.sigmoid(self.raw_widths[:, i, :]) * sigma_range_i + sigma_min_i

            mask = torch.exp(-0.5 * ((freq_axis - mu) / sigma)**2)
            mask = mask.unsqueeze(1)

            masked_spectrum = amp_spectrum * mask
            cnn_feat = self.spectrum_cnn(masked_spectrum).squeeze(-1)
            node_feat = self.fc_node(cnn_feat).unsqueeze(1)
            node_features.append(node_feat)

        nodes = torch.cat(node_features, dim=1)
        return nodes

# ==========================================
# 3. 图注意力机制 (GAT)
# ==========================================
class GraphAttentionLayer(nn.Module):
    def __init__(self, in_features, out_features, dropout=0.15, alpha=0.2):
        super(GraphAttentionLayer, self).__init__()
        self.W = nn.Linear(in_features, out_features, bias=False)
        self.a = nn.Linear(2 * out_features, 1, bias=False)
        self.leakyrelu = nn.LeakyReLU(alpha)
        self.dropout = nn.Dropout(dropout)

    def forward(self, h):
        B, N, F_dim = h.size()
        Wh = self.W(h)
        Wh_i = Wh.unsqueeze(2).expand(B, N, N, -1)
        Wh_j = Wh.unsqueeze(1).expand(B, N, N, -1)
        e = self.leakyrelu(self.a(torch.cat([Wh_i, Wh_j], dim=3)).squeeze(-1))

        attention = F.softmax(e, dim=-1)
        attention_drop = self.dropout(attention)

        h_prime = torch.bmm(attention_drop, Wh)
        aggregated_feature = torch.mean(h_prime, dim=1)
        return aggregated_feature, attention

# ==========================================
# 4. 原型分类器
# ==========================================
class PrototypeClassifier(nn.Module):
    def __init__(self, feature_dim, num_classes, temperature=1.0):
        super(PrototypeClassifier, self).__init__()
        self.num_classes = num_classes
        self.temperature = temperature
        self.prototypes = nn.Parameter(torch.randn(num_classes, feature_dim))
        nn.init.xavier_uniform_(self.prototypes)

    def forward(self, x):
        x_norm = F.normalize(x, p=2, dim=1)
        p_norm = F.normalize(self.prototypes, p=2, dim=1)
        cosine_sim = torch.matmul(x_norm, p_norm.t())
        logits = cosine_sim / self.temperature
        return logits, p_norm

    def get_ortho_loss(self, p_norm):
        identity = torch.eye(self.num_classes).to(p_norm.device)
        corr_matrix = torch.matmul(p_norm, p_norm.t())
        return torch.norm(corr_matrix - identity, p='fro') ** 2

# ==========================================
# 5. 域(工况)鉴别器
# ==========================================
class ConditionDiscriminator(nn.Module):
    def __init__(self, feature_dim, num_conditions=3):
        super(ConditionDiscriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_conditions)
        )

    def forward(self, x, alpha=1.0):
        x_rev = GradientReversalFunction.apply(x, alpha)
        return self.net(x_rev)

# ==========================================
# 6. 总体模型整合
# ==========================================
class TVNG_Spectrum_FaultModel(nn.Module):
    def __init__(self, num_classes=6, feature_dim=128, num_conditions=3):
        super(TVNG_Spectrum_FaultModel, self).__init__()
        self.physics_layer = DynamicSpectrumDecomposition(in_channels=1, out_channels=64, dropout_rate=0.15)
        self.gat = GraphAttentionLayer(in_features=128, out_features=feature_dim, dropout=0.15)
        self.classifier = PrototypeClassifier(feature_dim, num_classes)
        self.condition_discriminator = ConditionDiscriminator(feature_dim, num_conditions)

    def forward(self, x, cond, alpha=1.0):
        nodes = self.physics_layer(x, cond)
        features, attention_weights = self.gat(nodes)
        cls_logits, p_norm = self.classifier(features)
        cond_logits = self.condition_discriminator(features, alpha)
        return cls_logits, cond_logits, p_norm, features, attention_weights

# ==========================================
# 7. 多工况联合训练与验证主函数
# ==========================================
def train_multi_condition_model(train_x_paths, train_y_paths, val_x_paths, val_y_paths, run_idx=1):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[{device}] 正在加载多工况数据集...")

    train_dataset = MultiConditionDataset(train_x_paths, train_y_paths)
    val_dataset = MultiConditionDataset(val_x_paths, val_y_paths)

    if len(train_dataset) == 0:
        print("[错误] 未加载到任何数据！")
        return None, None

    batch_size = 64
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    num_classes = len(np.unique(train_dataset.labels))
    num_conditions = len(train_x_paths)

    model = TVNG_Spectrum_FaultModel(num_classes=num_classes, num_conditions=num_conditions).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    num_epochs = 160
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-5)

    criterion_cls = nn.CrossEntropyLoss()
    criterion_cond = nn.CrossEntropyLoss()
    lambda_ortho = 0.1
    lambda_cond = 0.4

    best_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    # 新增：用于记录全过程日志的列表
    training_history = []

    # 动态生成 CSV 表头
    csv_headers = ['Epoch', 'Train_Total_Loss', 'Train_Cls_Loss', 'Train_Cond_Loss', 'Train_Ortho_Loss', 'Val_Total_Acc']
    for c in range(num_conditions):
        csv_headers.append(f'Val_Acc_Cond_{c}')

    for epoch in range(num_epochs):
        model.train()

        # 归零各类损失
        epoch_cls_loss = 0.0
        epoch_cond_loss = 0.0
        epoch_ortho_loss = 0.0
        epoch_total_loss = 0.0

        for i, (data, fault_label, cond_label) in enumerate(train_loader):
            data, fault_label, cond_label = data.to(device), fault_label.to(device), cond_label.to(device)

            p = float(i + epoch * len(train_loader)) / num_epochs / len(train_loader)
            alpha = 2. / (1. + np.exp(-10 * p)) - 1

            cls_logits, cond_logits, p_norm, _, _ = model(data, cond_label, alpha)

            loss_cls = criterion_cls(cls_logits, fault_label)
            loss_cond = criterion_cond(cond_logits, cond_label)
            loss_ortho = model.classifier.get_ortho_loss(p_norm)

            total_loss = loss_cls + lambda_ortho * loss_ortho + lambda_cond * loss_cond

            optimizer.zero_grad()
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            # 累计各项损失
            epoch_cls_loss += loss_cls.item()
            epoch_cond_loss += loss_cond.item()
            epoch_ortho_loss += loss_ortho.item()
            epoch_total_loss += total_loss.item()

        # 计算该 Epoch 平均损失
        num_batches = len(train_loader)
        avg_cls_loss = epoch_cls_loss / num_batches
        avg_cond_loss = epoch_cond_loss / num_batches
        avg_ortho_loss = epoch_ortho_loss / num_batches
        avg_total_loss = epoch_total_loss / num_batches

        # ---------------- 验证阶段 ----------------
        model.eval()
        total_correct = 0
        total_samples = 0

        # 按工况追踪准确率
        cond_correct = {c: 0 for c in range(num_conditions)}
        cond_total = {c: 0 for c in range(num_conditions)}

        with torch.no_grad():
            for data, fault_label, cond_label in val_loader:
                data, fault_label, cond_label = data.to(device), fault_label.to(device), cond_label.to(device)

                cls_logits, _, _, _, _ = model(data, cond_label, alpha=0.0)
                _, predicted = torch.max(cls_logits.data, 1)

                total_samples += fault_label.size(0)
                total_correct += (predicted == fault_label).sum().item()

                # 细分统计不同工况的准确率
                for c in range(num_conditions):
                    mask = (cond_label == c)
                    cond_total[c] += mask.sum().item()
                    cond_correct[c] += (predicted[mask] == fault_label[mask]).sum().item()

        val_acc_total = 100 * total_correct / total_samples

        # 计算各工况验证准确率
        cond_acc_dict = {}
        for c in range(num_conditions):
            if cond_total[c] > 0:
                cond_acc_dict[c] = 100 * cond_correct[c] / cond_total[c]
            else:
                cond_acc_dict[c] = 0.0

        # 新增：保存当前 Epoch 指标到列表
        epoch_log = [
            epoch + 1,
            round(avg_total_loss, 4),
            round(avg_cls_loss, 4),
            round(avg_cond_loss, 4),
            round(avg_ortho_loss, 4),
            round(val_acc_total, 2)
        ]
        for c in range(num_conditions):
            epoch_log.append(round(cond_acc_dict[c], 2))

        training_history.append(epoch_log)

        # 打印日志 (避免刷屏，10 个 Epoch 打印一次，或者验证集达到目前最佳时)
        if (epoch + 1) % 10 == 0 or val_acc_total > best_acc:
            cond_acc_str = " | ".join([f"Cond {c}: {cond_acc_dict[c]:.2f}%" for c in range(num_conditions)])
            print(f"Epoch [{epoch+1:02d}/{num_epochs}] "
                  f"Total Loss: {avg_total_loss:.4f} (Cls: {avg_cls_loss:.4f}, Cond: {avg_cond_loss:.4f}) | "
                  f"Val Acc: {val_acc_total:.2f}% | {cond_acc_str}")

        if val_acc_total > best_acc:
            best_acc = val_acc_total
            best_model_wts = copy.deepcopy(model.state_dict())

        scheduler.step()

    print(f"\n✅ 第 {run_idx} 次训练结束！最高总体验证集准确率: {best_acc:.2f}%")
    model.load_state_dict(best_model_wts)

    # 动态保存每次的最佳模型
    model_save_path = f'best_model_run_{run_idx}.pth'
    torch.save(model.state_dict(), model_save_path)
    print(f"✅ 第 {run_idx} 次的最佳模型权重已保存为 '{model_save_path}'")

    # 新增：将训练日志完整写入 CSV 文件
    csv_save_path = f'training_log_run_{run_idx}.csv'
    with open(csv_save_path, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(csv_headers)
        writer.writerows(training_history)
    print(f"✅ 第 {run_idx} 次的完整训练日志已导出至 '{csv_save_path}'")

    return model, best_acc

In [ ]:
if __name__ == "__main__":
    train_x = ["xtra_0.npy", "xtra_1.npy", "xtra_2.npy"]
    train_y = ["ytra_0.npy", "ytra_1.npy", "ytra_2.npy"]
    val_x = ["xval_0.npy", "xval_1.npy", "xval_2.npy"]
    val_y = ["yval_0.npy", "yval_1.npy", "yval_2.npy"]

    num_runs = 30
    all_accuracies = []

    print(f"====== 即将开始 {num_runs} 次随机循环实验以评估模型鲁棒性 ======")

    for i in range(1, num_runs + 1):
        print(f"\n" + "-"*50)
        print(f"▶ 正在执行实验: Run {i}/{num_runs}")
        print("-"*50)

        # 赋予独立但可追踪的随机种子 (例如 42, 43, 44...)
        current_seed = 42 + i
        set_seed(current_seed)

        model, acc = train_multi_condition_model(train_x, train_y, val_x, val_y, run_idx=i)

        if acc is not None:
            all_accuracies.append(acc)

    # ==========================================
    # 最终的稳定性分析汇总报告
    # ==========================================
    if all_accuracies:
        print("\n" + "="*50)
        print("🏆 10次独立实验汇总报告")
        print("="*50)
        for idx, acc in enumerate(all_accuracies, 1):
            print(f"Run {idx:02d} 准确率: {acc:.2f}%")

        avg_acc = np.mean(all_accuracies)
        std_acc = np.std(all_accuracies)

        print("-" * 50)
        print(f"🔥 平均准确率 (Mean): {avg_acc:.2f}%")
        print(f"📉 标准差 (Std) : {std_acc:.4f}")
        print("=" * 50)
        print("💡 提示：用于后续分析脚本(如GAT/t-SNE/Mask)，您可选择准确率最高的那次权重文件 (如 best_model_run_X.pth)")

====== 即将开始 30 次随机循环实验以评估模型鲁棒性 ======

--------------------------------------------------
▶ 正在执行实验: Run 1/30
--------------------------------------------------
[cuda] 正在加载多工况数据集...
Epoch [01/160] Total Loss: 2.1172 (Cls: 1.6988, Cond: 1.0370) | Val Acc: 32.56% | Cond 0: 28.54% | Cond 1: 35.04% | Cond 2: 34.82%
Epoch [07/160] Total Loss: 1.8174 (Cls: 1.3878, Cond: 1.0347) | Val Acc: 32.74% | Cond 0: 37.89% | Cond 1: 24.50% | Cond 2: 34.82%
Epoch [10/160] Total Loss: 1.7598 (Cls: 1.3337, Cond: 1.0310) | Val Acc: 33.27% | Cond 0: 44.36% | Cond 1: 21.65% | Cond 2: 31.75%
Epoch [12/160] Total Loss: 1.6931 (Cls: 1.2773, Cond: 1.0076) | Val Acc: 39.04% | Cond 0: 46.28% | Cond 1: 35.04% | Cond 2: 34.54%
Epoch [13/160] Total Loss: 1.6796 (Cls: 1.2666, Cond: 1.0047) | Val Acc: 42.41% | Cond 0: 45.56% | Cond 1: 47.01% | Cond 2: 34.26%
Epoch [14/160] Total Loss: 1.6535 (Cls: 1.2394, Cond: 1.0092) | Val Acc: 53.33% | Cond 0: 63.31% | Cond 1: 48.15% | Cond 2: 46.80%
Epoch [15/160] Total Loss: 1.6401

KeyboardInterrupt: 

In [ ]:
import torch
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
import os
from torch.utils.data import DataLoader

def export_tsne_to_csv(model_path, val_x_paths, val_y_paths, save_dir='./TSNE_Results'):
    os.makedirs(save_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 [Step 1] 正在加载模型提取特征 (Device: {device})...")

    # 1. 加载数据和模型 (确保 MultiConditionDataset 和 TVNG_Spectrum_FaultModel 已定义)
    val_dataset = MultiConditionDataset(val_x_paths, val_y_paths)
    val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

    model = TVNG_Spectrum_FaultModel(num_classes=6, feature_dim=128, num_conditions=3).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # 2. 提取特征
    all_features, all_fault_labels, all_cond_labels = [], [], []
    with torch.no_grad():
        for data, fault_label, cond_label in val_loader:
            # 评估阶段 alpha=0.0
            _, _, _, features, _ = model(data.to(device), cond_label.to(device), alpha=0.0)
            all_features.append(features.cpu().numpy())
            all_fault_labels.append(fault_label.numpy())
            all_cond_labels.append(cond_label.numpy())

    features_mat = np.concatenate(all_features, axis=0)
    fault_arr = np.concatenate(all_fault_labels, axis=0)
    cond_arr = np.concatenate(all_cond_labels, axis=0)

    # 3. t-SNE 降维
    print("🧮 [Step 2] 正在运行 t-SNE 流形降维运算 (将 128 维降至 2 维)...")
    tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto', n_iter=1500, random_state=42)
    features_2d = tsne.fit_transform(features_mat)

    # 4. 打包为 DataFrame 并导出 CSV
    df = pd.DataFrame({
        'tSNE_1': features_2d[:, 0],
        'tSNE_2': features_2d[:, 1],
        'Fault_Class': fault_arr,
        'Condition_RPM': cond_arr
    })

    csv_path = os.path.join(save_dir, 'tSNE_Features_Data.csv')
    df.to_csv(csv_path, index=False)
    print(f"✅ 数据已成功导出至: {csv_path}！你可以把它下载到本地电脑上进行后续绘图。")

if __name__ == "__main__":
    # 使用你的验证集数据路径
    val_x = ["xval_0.npy", "xval_1.npy", "xval_2.npy"]
    val_y = ["yval_0.npy", "yval_1.npy", "yval_2.npy"]
    model_weight_path = 'best_model_run_2.pth'
    export_tsne_to_csv(model_weight_path, val_x, val_y)

🚀 [Step 1] 正在加载模型提取特征 (Device: cuda)...
🧮 [Step 2] 正在运行 t-SNE 流形降维运算 (将 128 维降至 2 维)...


/usr/local/lib/python3.12/dist-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


✅ 数据已成功导出至: ./TSNE_Results/tSNE_Features_Data.csv！你可以把它下载到本地电脑上进行后续绘图。


In [ ]:
import torch
import numpy as np
import pandas as pd
import os

def export_masks_to_csv(model_path, save_dir='./Mask_Analysis_Results'):
    """
    加载最佳模型，提取物理层的动态掩码参数，并将曲线数据导出为 CSV。
    """
    os.makedirs(save_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 [Step 1] 正在加载模型权重提取物理掩码数据: {model_path} (Device: {device})")

    # 1. 初始化并加载你的 SOTA 模型
    # 注意：运行此代码前，请确保 TVNG_Spectrum_FaultModel 已在代码中定义
    model = TVNG_Spectrum_FaultModel(num_classes=6, feature_dim=128, num_conditions=3).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # 2. 提取并分离网络中的可学习参数
    physics_layer = model.physics_layer
    mu_shift = physics_layer.mu_shift[0, :, 0].detach().cpu()       # Shape: [5]
    raw_widths = physics_layer.raw_widths[0, :, 0].detach().cpu()   # Shape: [5]
    sigma_mins = physics_layer.sigma_mins[0, :, 0].cpu()            # Shape: [5]
    sigma_ranges = physics_layer.sigma_ranges[0, :, 0].cpu()        # Shape: [5]

    # 3. 物理系统参数与高分辨率频率轴
    f_nyq = 1600.0
    freq_bins = 2000  # 使用超高分辨率确保曲线平滑
    freq_axis_norm = torch.linspace(0, 1, freq_bins)
    freq_axis_hz = freq_axis_norm.numpy() * f_nyq

    rpms = [900.0, 1350.0, 1800.0]
    node_names = ['Baseline Drift', 'Cage (FTF)', 'Shaft (1X/2X)', 'Fault Freq Group', 'High-Freq Resonance']

    # 4. 遍历三种工况，计算曲线并导出
    for cond in range(3):
        fr = rpms[cond] / 60.0

        # [核心] 复现网络前向传播中的物理动力学引擎
        base_mu = torch.zeros(5)
        base_mu[0] = 0.001
        base_mu[1] = 0.4 * fr / f_nyq
        base_mu[2] = 1.65 * fr / f_nyq
        base_mu[3] = 4.5 * fr / f_nyq
        base_mu[4] = 0.18

        # 应用网络学习到的容差补偿与异质带宽
        mu_raw = base_mu + torch.tanh(mu_shift) * 0.015
        learned_mu = torch.clamp(mu_raw, min=0.0, max=1.0)
        learned_sigma = torch.sigmoid(raw_widths) * sigma_ranges + sigma_mins

        # 准备导出数据的 DataFrame
        df_export = pd.DataFrame({'Frequency_Hz': freq_axis_hz})

        # 逐节点计算高斯掩码并存入 DataFrame
        for i in range(5):
            mask = torch.exp(-0.5 * ((freq_axis_norm - learned_mu[i]) / learned_sigma[i])**2).numpy()
            col_name = f'Node_{i}_{node_names[i].replace(" ", "_").replace("/", "_")}'
            df_export[col_name] = mask

        # 导出当前工况的数据到 CSV
        csv_path = os.path.join(save_dir, f'Mask_Data_Cond_{cond}_{int(rpms[cond])}RPM.csv')
        df_export.to_csv(csv_path, index=False)
        print(f"✅ 工况 {cond} ({int(rpms[cond])} RPM) 掩码数据已导出: {csv_path}")

if __name__ == "__main__":
    model_weight_path = 'best_model_run_2.pth' # 你的权重文件
    if os.path.exists(model_weight_path):
        export_masks_to_csv(model_weight_path)
    else:
        print(f"❌ 找不到权重文件 '{model_weight_path}'，请检查路径是否正确！")

🚀 [Step 1] 正在加载模型权重提取物理掩码数据: best_model_run_2.pth (Device: cuda)
✅ 工况 0 (900 RPM) 掩码数据已导出: ./Mask_Analysis_Results/Mask_Data_Cond_0_900RPM.csv
✅ 工况 1 (1350 RPM) 掩码数据已导出: ./Mask_Analysis_Results/Mask_Data_Cond_1_1350RPM.csv
✅ 工况 2 (1800 RPM) 掩码数据已导出: ./Mask_Analysis_Results/Mask_Data_Cond_2_1800RPM.csv


In [ ]:
#GAT COND
import torch
import numpy as np
import pandas as pd
import os
from torch.utils.data import DataLoader

def export_gat_attention(model_path, val_x_paths, val_y_paths, save_dir='./Graph_Analysis'):
    os.makedirs(save_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 [Step 1] 正在加载模型提取图注意力权重 (Device: {device})...")

    # 1. 加载数据和模型
    val_dataset = MultiConditionDataset(val_x_paths, val_y_paths)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

    model = TVNG_Spectrum_FaultModel(num_classes=6, feature_dim=128, num_conditions=3).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # 用于累加不同工况下的注意力矩阵
    attention_sums = {0: np.zeros((5, 5)), 1: np.zeros((5, 5)), 2: np.zeros((5, 5))}
    condition_counts = {0: 0, 1: 0, 2: 0}

    # 2. 提取特征与注意力权重
    with torch.no_grad():
        for data, fault_label, cond_label in val_loader:
            data, cond_label = data.to(device), cond_label.to(device)
            # 获取 GAT 输出的注意力矩阵 attention_weights: Shape [Batch, 5, 5]
            _, _, _, _, attention_weights = model(data, cond_label, alpha=0.0)

            att_np = attention_weights.cpu().numpy()
            cond_np = cond_label.cpu().numpy()

            # 按工况累加
            for b_idx in range(att_np.shape[0]):
                c = cond_np[b_idx]
                attention_sums[c] += att_np[b_idx]
                condition_counts[c] += 1

    # 3. 计算平均值并导出 CSV
    node_names = ['Baseline', 'Cage', 'Shaft', 'Fault Freq', 'Resonance']
    for c in range(3):
        if condition_counts[c] > 0:
            avg_attention = attention_sums[c] / condition_counts[c]
            df = pd.DataFrame(avg_attention, index=node_names, columns=node_names)

            csv_path = os.path.join(save_dir, f'GAT_Attention_Cond_{c}.csv')
            df.to_csv(csv_path)
            print(f"✅ 工况 {c} 的注意力矩阵已导出至: {csv_path}")

if __name__ == "__main__":
    val_x = ["xval_0.npy", "xval_1.npy", "xval_2.npy"]
    val_y = ["yval_0.npy", "yval_1.npy", "yval_2.npy"]
    export_gat_attention('best_model_run_2.pth', val_x, val_y)

🚀 [Step 1] 正在加载模型提取图注意力权重 (Device: cuda)...
✅ 工况 0 的注意力矩阵已导出至: ./Graph_Analysis/GAT_Attention_Cond_0.csv
✅ 工况 1 的注意力矩阵已导出至: ./Graph_Analysis/GAT_Attention_Cond_1.csv
✅ 工况 2 的注意力矩阵已导出至: ./Graph_Analysis/GAT_Attention_Cond_2.csv


In [ ]:
#GAT CLASS
import torch
import numpy as np
import pandas as pd
import os
from torch.utils.data import DataLoader

def export_gat_attention_by_fault(model_path, val_x_paths, val_y_paths, save_dir='./Graph_Analysis', target_cond=None):
    """
    提取按故障类别划分的 GAT 注意力权重。
    参数:
    - target_cond:
        None -> 提取并平均所有工况的数据 (全局视角)
        0, 1, 2 -> 仅提取特定工况的数据 (如 0 代表 900 RPM)
    """
    os.makedirs(save_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 动态生成保存文件的前缀
    cond_name = "All_Conditions" if target_cond is None else f"Cond_{target_cond}"
    print(f"🚀 [Step 1] 正在提取按【故障类别】划分的图注意力权重 (数据范围: {cond_name})...")

    val_dataset = MultiConditionDataset(val_x_paths, val_y_paths)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

    model = TVNG_Spectrum_FaultModel(num_classes=6, feature_dim=128, num_conditions=3).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    attention_sums = {i: np.zeros((5, 5)) for i in range(6)}
    fault_counts = {i: 0 for i in range(6)}

    with torch.no_grad():
        for data, fault_label, cond_label in val_loader:
            data, cond_label = data.to(device), cond_label.to(device)
            _, _, _, _, attention_weights = model(data, cond_label, alpha=0.0)

            att_np = attention_weights.cpu().numpy()
            fault_np = fault_label.cpu().numpy()
            cond_np = cond_label.cpu().numpy() # 提取工况标签用于筛选

            for b_idx in range(att_np.shape[0]):
                # [核心逻辑]: 如果指定了特定工况，且当前样本不属于该工况，则直接跳过
                if target_cond is not None and cond_np[b_idx] != target_cond:
                    continue

                f = fault_np[b_idx]
                attention_sums[f] += att_np[b_idx]
                fault_counts[f] += 1

    node_names = ['Baseline', 'Cage', 'Shaft', 'Fault Freq', 'Resonance']
    for f in range(6):
        if fault_counts[f] > 0:
            avg_attention = attention_sums[f] / fault_counts[f]
            df = pd.DataFrame(avg_attention, index=node_names, columns=node_names)
            # 在文件名中加入工况标识，避免覆盖
            csv_path = os.path.join(save_dir, f'GAT_Attention_Fault_{f}_{cond_name}.csv')
            df.to_csv(csv_path)

    print(f"✅ [{cond_name}] 范围内 6 种故障类别的注意力矩阵已成功导出！")

if __name__ == "__main__":
    val_x = ["xval_0.npy", "xval_1.npy", "xval_2.npy"]
    val_y = ["yval_0.npy", "yval_1.npy", "yval_2.npy"]
    model_weight = 'best_model_run_2.pth'

    # 示例用法：
    # 1. 提取所有工况融合的本质特征矩阵
    export_gat_attention_by_fault(model_weight, val_x, val_y, target_cond=None)

    # 2. 提取 1800 RPM (Cond 2) 时的特定矩阵
    export_gat_attention_by_fault(model_weight, val_x, val_y, target_cond=1)

🚀 [Step 1] 正在提取按【故障类别】划分的图注意力权重 (数据范围: All_Conditions)...
✅ [All_Conditions] 范围内 6 种故障类别的注意力矩阵已成功导出！
🚀 [Step 1] 正在提取按【故障类别】划分的图注意力权重 (数据范围: Cond_1)...
✅ [Cond_1] 范围内 6 种故障类别的注意力矩阵已成功导出！


In [ ]:
#混淆矩阵
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import os
from torch.utils.data import DataLoader

def export_confusion_matrix_to_csv(model_path, val_x_paths, val_y_paths, save_dir='./CM_Results'):
    os.makedirs(save_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 [Step 1] 正在加载模型并计算混淆矩阵 (Device: {device})...")

    # 1. 加载验证集和模型
    val_dataset = MultiConditionDataset(val_x_paths, val_y_paths)
    val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

    model = TVNG_Spectrum_FaultModel(num_classes=6, feature_dim=128, num_conditions=3).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    all_preds = []
    all_trues = []

    # 2. 收集预测结果
    with torch.no_grad():
        for data, fault_label, cond_label in val_loader:
            data = data.to(device)
            cond_label = cond_label.to(device)
            # 推理阶段 alpha 置 0
            cls_logits, _, _, _, _ = model(data, cond_label, alpha=0.0)

            # 获取最大概率的索引作为预测类别
            _, preds = torch.max(cls_logits, 1)

            all_preds.extend(preds.cpu().numpy())
            all_trues.extend(fault_label.numpy())

    # 3. 计算混淆矩阵并导出
    class_names = ['Normal', 'Inner Race', 'Outer Race', 'Ball', 'Cage', 'Combined']
    cm = confusion_matrix(all_trues, all_preds)

    # 将 NumPy 矩阵转为带有行列名称的 DataFrame
    df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)

    csv_path = os.path.join(save_dir, 'Confusion_Matrix_Data.csv')
    df_cm.to_csv(csv_path)
    print(f"✅ 混淆矩阵数据已成功导出至: {csv_path}")

if __name__ == "__main__":
    val_x = ["xval_0.npy", "xval_1.npy", "xval_2.npy"]
    val_y = ["yval_0.npy", "yval_1.npy", "yval_2.npy"]
    export_confusion_matrix_to_csv('best_model_run_2.pth', val_x, val_y)